In [ ]:
%pip install -q lyricsgenius python-dotenv pandas

In [2]:
import os
from pathlib import Path

import pandas as pd
import lyricsgenius
from dotenv import load_dotenv

load_dotenv(Path('..') / '.env')

GENIUS_ACCESS_TOKEN = os.getenv('GENIUS_ACCESS_TOKEN')

if not GENIUS_ACCESS_TOKEN:
    raise ValueError('Missing GENIUS_ACCESS_TOKEN in .env')

## 1) Initialise Genius client

In [3]:
genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True
)
genius.verbose = False
print('Genius client ready.')

Genius client ready.


---
## Build per-region top songs from Spotify Charts

Reads the downloaded Spotify Charts CSVs and keeps only each region's top-ranked song.
No language detection is applied at this stage.

1. Merge all songs into single df with just `artist`, `title`, `spotify_uri`
2. Fetch lyrics into `lyrics`

Output columns: `rank`, `artist`, `title`, `region`, `spotify_uri`

In [4]:
from pathlib import Path
import pandas as pd

print('Top-song extraction helpers loaded.')

Top-song extraction helpers loaded.


In [ ]:
from pathlib import Path
import pandas as pd

RAW_BASE = Path('..') / 'data' / 'raw'
RAW_DIR = RAW_BASE

OUTPUT_PATH = Path('..') / 'data' / 'processed' / 'titles.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

FILE_REGION_MAP = {
    'regional-us-weekly-2026-03-05.csv': 'USA',
    'regional-co-weekly-2026-03-05.csv': 'Colombia',
    'regional-tw-weekly-2026-03-05.csv': 'Taiwan',
    'regional-global-weekly-2026-03-05.csv': 'Global',
}

dfs = []
for filename, region in FILE_REGION_MAP.items():
    df = pd.read_csv(RAW_DIR / filename)
    df = df.rename(columns={'artist_names': 'artist', 'track_name': 'title'})
    df['rank'] = pd.to_numeric(df['rank'], errors='coerce')
    df = df.dropna(subset=['rank'])

    df = df.assign(
        region=region,
        spotify_uri=df['uri'].str.replace('spotify:track:', '', regex=False)
    )[["rank", "artist", "title", "region", "spotify_uri"]]

    dfs.append(df)

titles_df = (
    pd.concat(dfs)
    .sort_values(['region', 'rank'])
    .reset_index(drop=True)
)

titles_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved to: {OUTPUT_PATH}')

Saved to: /Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv


In [ ]:
titles_df

---
## Genius lyrics fetch → chunked writes to `lyrics.csv`

Fetches lyrics for each region's top song and writes incrementally to:
`data/processed/lyrics.csv`

- Appends every chunk (safe to resume)
- Skips songs already present in output
- Avoids losing all progress if a run fails

In [ ]:
import time
from pathlib import Path
import pandas as pd

LYRICS_OUT = Path('..') / 'data' / 'processed' / 'lyrics.csv'

CHUNK_SIZE = 10       # write to disk every N fetched songs
SLEEP_BETWEEN = 0.4   # seconds between Genius API calls

LYRICS_OUT.parent.mkdir(parents=True, exist_ok=True)

required_cols = [
    'rank', 'artist', 'title', 'region', 'spotify_uri', 'lyrics'
]

In [ ]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    """Return lyrics string or empty string on failure."""
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f'  [warn] {title!r} by {artist!r}: {e}')
    return ''

def append_chunk(rows_chunk: list[dict], out_path: Path) -> None:
    """Append a list of row dicts to the CSV, writing the header only if the file doesn't exist yet."""
    if not rows_chunk:
        return
    chunk_df = pd.DataFrame(rows_chunk)[required_cols]
    write_header = not out_path.exists()
    chunk_df.to_csv(out_path, mode='a', header=write_header, index=False)

# ── Determine which songs still need lyrics ──────────────────────────────────
if LYRICS_OUT.exists():
    existing = pd.read_csv(LYRICS_OUT)
    fetched_uris = set(existing['spotify_uri'].astype(str))
    print(f'Found {len(fetched_uris)} songs already in {LYRICS_OUT.name}')
else:
    fetched_uris = set()
    print(f'{LYRICS_OUT.name} not found — starting fresh')

pending = titles_df[~titles_df['spotify_uri'].astype(str).isin(fetched_uris)].reset_index(drop=True)
total = len(pending)

if total == 0:
    print('All songs already fetched — nothing to do.')
else:
    print(f'Songs to fetch: {total}')
    rows_buffer = []
    for i, row in pending.iterrows():
        lyrics = fetch_lyrics_genius(genius, row['title'], row['artist'])
        rows_buffer.append({**row.to_dict(), 'lyrics': lyrics})

        status = 'ok' if lyrics else 'missing'
        print(f'[{i+1}/{total}] {row["region"]:12s} | {status} | {row["title"][:50]}')

        if len(rows_buffer) >= CHUNK_SIZE:
            append_chunk(rows_buffer, LYRICS_OUT)
            print(f'  \u2192 flushed {len(rows_buffer)} rows')
            rows_buffer = []

        time.sleep(SLEEP_BETWEEN)

    # flush remaining rows
    if rows_buffer:
        append_chunk(rows_buffer, LYRICS_OUT)
        print(f'  \u2192 flushed final {len(rows_buffer)} rows')

    print(f'Done. Output: {LYRICS_OUT}')

In [ ]:
lyrics_df = pd.read_csv(LYRICS_OUT)

print(f'Saved rows: {len(lyrics_df)}')
print(f'Output path: {LYRICS_OUT}')

coverage = (lyrics_df
    .assign(has_lyrics=lyrics_df['lyrics'].fillna('').str.strip().ne(''))
    .groupby('region')['has_lyrics']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'with_lyrics', 'count': 'total'})
)
coverage['pct'] = (coverage['with_lyrics'] / coverage['total'] * 100).round(1)
print('\nLyrics coverage by region:')
print(coverage.to_string())

lyrics_df.head(10)